# Fraud Detection - Feature Engineering

This notebook explores and engineers features for fraud detection.

## Objectives
- Load and explore transaction data
- Create meaningful features for fraud detection
- Analyze feature importance and correlations
- Save processed features for model training

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

## 1. Load Data

In [ ]:
# Load transaction data
# In a real scenario, this would be your actual transaction data
data_path = '../data/processed/processed_data.csv'

try:
    df = pd.read_csv(data_path)
    print(f"Loaded data with shape: {df.shape}")
except FileNotFoundError:
    print("No processed data found. Generating sample data for demonstration...")
    
    # Generate sample data for demonstration
    np.random.seed(42)
    n_samples = 10000
    
    df = pd.DataFrame({
        'transaction_id': [f'txn_{i:06d}' for i in range(n_samples)],
        'user_id': [f'user_{i:04d}' for i in np.random.randint(1, 1000, n_samples)],
        'amount': np.random.exponential(100, n_samples),
        'timestamp': pd.date_range('2024-01-01', periods=n_samples, freq='1min'),
        'merchant_category': np.random.randint(1, 10, n_samples),
        'merchant_id': [f'merch_{i:04d}' for i in np.random.randint(1, 500, n_samples)],
        'customer_age': np.random.randint(18, 80, n_samples),
        'customer_tenure': np.random.randint(0, 365, n_samples),
        'is_online': np.random.choice([0, 1], n_samples, p=[0.3, 0.7]),
        'is_foreign': np.random.choice([0, 1], n_samples, p=[0.9, 0.1]),
        'device_type': np.random.choice(['mobile', 'desktop', 'tablet'], n_samples),
        'distance_from_home': np.random.exponential(50, n_samples),
        'is_fraud': np.random.choice([0, 1], n_samples, p=[0.95, 0.05])
    })
    
    print(f"Generated sample data with shape: {df.shape}")

print("\nFirst few rows:")
df.head()

## 2. Exploratory Data Analysis

In [ ]:
# Basic statistics
print("Data Info:")
df.info()

print("\nDescriptive Statistics:")
df.describe()

In [ ]:
# Fraud distribution
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
df['is_fraud'].value_counts().plot(kind='bar')
plt.title('Fraud Distribution')
plt.xlabel('Is Fraud')
plt.ylabel('Count')
plt.xticks(rotation=0)

plt.subplot(1, 2, 2)
fraud_rate = df['is_fraud'].mean() * 100
plt.pie([100-fraud_rate, fraud_rate], labels=['Not Fraud', 'Fraud'], autopct='%1.2f%%')
plt.title('Fraud Rate')

plt.tight_layout()
plt.show()

print(f"Overall fraud rate: {fraud_rate:.2f}%")

In [ ]:
# Amount distribution by fraud status
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
sns.histplot(data=df, x='amount', hue='is_fraud', bins=50, alpha=0.7)
plt.title('Amount Distribution by Fraud Status')
plt.xlabel('Amount')
plt.ylabel('Frequency')

plt.subplot(1, 3, 2)
sns.boxplot(data=df, x='is_fraud', y='amount')
plt.title('Amount Box Plot by Fraud Status')
plt.xlabel('Is Fraud')
plt.ylabel('Amount')

plt.subplot(1, 3, 3)
amount_stats = df.groupby('is_fraud')['amount'].agg(['mean', 'median', 'std'])
amount_stats.plot(kind='bar')
plt.title('Amount Statistics by Fraud Status')
plt.xlabel('Is Fraud')
plt.ylabel('Amount')
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

print(amount_stats)

## 3. Feature Engineering

In [ ]:
# Create a copy for feature engineering
df_features = df.copy()

# Time-based features
if 'timestamp' in df_features.columns:
    df_features['timestamp'] = pd.to_datetime(df_features['timestamp'])
    df_features['hour'] = df_features['timestamp'].dt.hour
    df_features['day_of_week'] = df_features['timestamp'].dt.dayofweek
    df_features['month'] = df_features['timestamp'].dt.month
    df_features['is_weekend'] = df_features['day_of_week'].isin([5, 6]).astype(int)
    df_features['is_night'] = ((df_features['hour'] >= 22) | (df_features['hour'] <= 6)).astype(int)
    df_features['is_business_hours'] = ((df_features['hour'] >= 9) & (df_features['hour'] <= 17)).astype(int)

print("Time-based features created:")
time_features = ['hour', 'day_of_week', 'month', 'is_weekend', 'is_night', 'is_business_hours']
for feature in time_features:
    if feature in df_features.columns:
        print(f"  - {feature}")

In [ ]:
# Amount-based features
amount_col = 'amount'
if amount_col in df_features.columns:
    df_features['amount_log'] = np.log1p(df_features[amount_col])
    df_features['amount_sqrt'] = np.sqrt(df_features[amount_col])
    df_features['amount_squared'] = df_features[amount_col] ** 2
    
    # Amount bins
    df_features['amount_bin'] = pd.cut(df_features[amount_col], 
                                       bins=[0, 10, 50, 100, 500, np.inf], 
                                       labels=['very_low', 'low', 'medium', 'high', 'very_high'])
    
    # High amount indicator
    df_features['is_high_amount'] = (df_features[amount_col] > 100).astype(int)
    df_features['is_very_high_amount'] = (df_features[amount_col] > 500).astype(int)

print("Amount-based features created:")
amount_features = ['amount_log', 'amount_sqrt', 'amount_squared', 'amount_bin', 'is_high_amount', 'is_very_high_amount']
for feature in amount_features:
    if feature in df_features.columns:
        print(f"  - {feature}")

In [ ]:
# Customer behavior features
if 'customer_age' in df_features.columns and amount_col in df_features.columns:
    df_features['amount_per_age'] = df_features[amount_col] / (df_features['customer_age'] + 1)
    df_features['is_young_customer'] = (df_features['customer_age'] < 25).astype(int)
    df_features['is_senior_customer'] = (df_features['customer_age'] > 65).astype(int)

if 'customer_tenure' in df_features.columns:
    df_features['tenure_years'] = df_features['customer_tenure'] / 365.25
    df_features['is_new_customer'] = (df_features['customer_tenure'] < 30).astype(int)
    df_features['is_loyal_customer'] = (df_features['customer_tenure'] > 365).astype(int)

print("Customer behavior features created:")
customer_features = ['amount_per_age', 'is_young_customer', 'is_senior_customer', 
                    'tenure_years', 'is_new_customer', 'is_loyal_customer']
for feature in customer_features:
    if feature in df_features.columns:
        print(f"  - {feature}")

In [ ]:
# Geographic and device features
if 'distance_from_home' in df_features.columns:
    df_features['distance_log'] = np.log1p(df_features['distance_from_home'])
    df_features['is_far_from_home'] = (df_features['distance_from_home'] > 100).astype(int)
    df_features['is_very_far_from_home'] = (df_features['distance_from_home'] > 500).astype(int)

if 'device_type' in df_features.columns:
    df_features['is_mobile'] = (df_features['device_type'] == 'mobile').astype(int)
    df_features['is_desktop'] = (df_features['device_type'] == 'desktop').astype(int)

print("Geographic and device features created:")
geo_device_features = ['distance_log', 'is_far_from_home', 'is_very_far_from_home', 'is_mobile', 'is_desktop']
for feature in geo_device_features:
    if feature in df_features.columns:
        print(f"  - {feature}")

In [ ]:
# Risk-based features
if 'merchant_category' in df_features.columns:
    # High-risk merchant categories (example)
    high_risk_categories = [1, 5, 8]  # These would be determined from domain knowledge
    df_features['high_risk_merchant'] = df_features['merchant_category'].isin(high_risk_categories).astype(int)

# Composite risk features
risk_features = []
if 'is_high_amount' in df_features.columns:
    risk_features.append('is_high_amount')
if 'is_night' in df_features.columns:
    risk_features.append('is_night')
if 'is_foreign' in df_features.columns:
    risk_features.append('is_foreign')
if 'is_far_from_home' in df_features.columns:
    risk_features.append('is_far_from_home')

if risk_features:
    df_features['risk_score'] = df_features[risk_features].sum(axis=1)
    df_features['high_risk_transaction'] = (df_features['risk_score'] >= 2).astype(int)
    print(f"Risk score created from {len(risk_features)} risk factors")
    print(f"  Risk factors: {risk_features}")

## 4. Feature Analysis

In [ ]:
# Analyze feature correlation with fraud
numeric_features = df_features.select_dtypes(include=[np.number]).columns
correlations = df_features[numeric_features].corr()['is_fraud'].sort_values(ascending=False)

print("Feature correlations with fraud:")
print(correlations.head(15))

# Plot top correlations
plt.figure(figsize=(12, 8))
top_features = correlations.abs().head(15).index
correlations[top_features].plot(kind='barh')
plt.title('Top 15 Features Correlated with Fraud')
plt.xlabel('Correlation with Fraud')
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance by fraud rate
categorical_features = df_features.select_dtypes(include=['object', 'category']).columns
binary_features = [col for col in df_features.columns if df_features[col].nunique() == 2 and col != 'is_fraud']

print("Fraud rate by categorical features:")
for feature in binary_features[:10]:  # Show first 10
    if feature in df_features.columns:
        fraud_rate = df_features.groupby(feature)['is_fraud'].mean()
        print(f"\n{feature}:")
        for value in sorted(df_features[feature].unique()):
            rate = fraud_rate.get(value, 0)
            count = df_features[df_features[feature] == value].shape[0]
            print(f"  {value}: {rate:.3f} ({count} transactions)")

## 5. Data Visualization

In [ ]:
# Create a comprehensive visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# 1. Fraud by hour
if 'hour' in df_features.columns:
    hourly_fraud = df_features.groupby('hour')['is_fraud'].mean()
    axes[0, 0].plot(hourly_fraud.index, hourly_fraud.values, marker='o')
    axes[0, 0].set_title('Fraud Rate by Hour of Day')
    axes[0, 0].set_xlabel('Hour')
    axes[0, 0].set_ylabel('Fraud Rate')
    axes[0, 0].grid(True)

# 2. Fraud by day of week
if 'day_of_week' in df_features.columns:
    dow_fraud = df_features.groupby('day_of_week')['is_fraud'].mean()
    dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
    axes[0, 1].bar(range(7), [dow_fraud.get(i, 0) for i in range(7)])
    axes[0, 1].set_title('Fraud Rate by Day of Week')
    axes[0, 1].set_xlabel('Day of Week')
    axes[0, 1].set_ylabel('Fraud Rate')
    axes[0, 1].set_xticks(range(7))
    axes[0, 1].set_xticklabels(dow_labels)

# 3. Amount distribution
axes[0, 2].hist([df_features[df_features['is_fraud'] == 0][amount_col], 
                   df_features[df_features['is_fraud'] == 1][amount_col]], 
                   bins=50, alpha=0.7, label=['Not Fraud', 'Fraud'])
axes[0, 2].set_title('Amount Distribution')
axes[0, 2].set_xlabel('Amount')
axes[0, 2].set_ylabel('Frequency')
axes[0, 2].legend()

# 4. Risk score distribution
if 'risk_score' in df_features.columns:
    axes[1, 0].hist([df_features[df_features['is_fraud'] == 0]['risk_score'], 
                      df_features[df_features['is_fraud'] == 1]['risk_score']], 
                      bins=20, alpha=0.7, label=['Not Fraud', 'Fraud'])
    axes[1, 0].set_title('Risk Score Distribution')
    axes[1, 0].set_xlabel('Risk Score')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].legend()

# 5. Customer age vs fraud
if 'customer_age' in df_features.columns:
    age_fraud = df_features.groupby(pd.cut(df_features['customer_age'], bins=5))['is_fraud'].mean()
    age_fraud.plot(kind='bar', ax=axes[1, 1])
    axes[1, 1].set_title('Fraud Rate by Customer Age')
    axes[1, 1].set_xlabel('Customer Age')
    axes[1, 1].set_ylabel('Fraud Rate')
    axes[1, 1].tick_params(axis='x', rotation=45)

# 6. Distance from home vs fraud
if 'distance_from_home' in df_features.columns:
    distance_fraud = df_features.groupby(pd.cut(df_features['distance_from_home'], bins=5))['is_fraud'].mean()
    distance_fraud.plot(kind='bar', ax=axes[1, 2])
    axes[1, 2].set_title('Fraud Rate by Distance from Home')
    axes[1, 2].set_xlabel('Distance from Home')
    axes[1, 2].set_ylabel('Fraud Rate')
    axes[1, 2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 6. Save Processed Features

In [ ]:
# Prepare final feature set
# Select only numeric features for modeling
final_features = df_features.select_dtypes(include=[np.number])

# Remove highly correlated features (optional)
correlation_matrix = final_features.corr().abs()
upper_triangle = correlation_matrix.where(np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool))
highly_correlated = [column for column in upper_triangle.columns if any(upper_triangle[column] > 0.95)]

print(f"Highly correlated features to remove: {highly_correlated}")
final_features = final_features.drop(columns=highly_correlated)

print(f"\nFinal feature set shape: {final_features.shape}")
print(f"Features: {list(final_features.columns)}")

# Save processed features
output_path = '../data/processed/engineered_features.csv'
os.makedirs('../data/processed', exist_ok=True)
final_features.to_csv(output_path, index=False)

print(f"\nProcessed features saved to: {output_path}")

# Save feature list for reference
feature_list_path = '../data/processed/feature_list.txt'
with open(feature_list_path, 'w') as f:
    for feature in final_features.columns:
        f.write(f"{feature}\n")

print(f"Feature list saved to: {feature_list_path}")

## 7. Summary

### Key Findings:
1. **Temporal Patterns**: Fraud rates vary by time of day and day of week
2. **Amount Patterns**: Higher amounts are generally associated with higher fraud rates
3. **Customer Behavior**: New customers and certain age groups show different fraud patterns
4. **Geographic Patterns**: Distance from home correlates with fraud likelihood
5. **Composite Risk**: Multiple risk factors combined provide better fraud detection

### Engineered Features:
- **Time-based**: Hour, day of week, weekend/night indicators
- **Amount-based**: Log transformations, amount bins, high amount indicators
- **Customer**: Age-based features, tenure indicators
- **Geographic**: Distance transformations, location-based risk
- **Composite**: Risk scores combining multiple factors

### Next Steps:
1. Use engineered features for model training
2. Validate feature importance with model-based methods
3. Monitor feature drift in production
4. Iterate on feature engineering based on model performance